# 03 · SARIMAX Models

Fits four SARIMAX variants — one per predictor strategy — and evaluates multi-step demand forecasts at H ∈ {1, 3, 7, 30, 90, 180} days on the held-out test set.

## Architecture

All four variants use a **two-stage approach** for consistency and to isolate the interpretability contribution:

1. **Stage 1 — Exogenous regression:** Fit a regularised (or filtered) linear model on exogenous predictors to capture the structural demand component.
   `ŷ_exog[t] = X[t] · β`

2. **Stage 2 — SARIMA on residuals:** Fit SARIMA(p,d,q)(P,D,Q)_s on the residual `e[t] = y[t] − ŷ_exog[t]` to capture temporal autocorrelation.

3. **Combined forecast:** `ŷ[t+h] = ŷ_exog[t+h] + ê_SARIMA[t+h | t]`

| Variant | Stage 1 model | Features |
|---|---|---|
| SARIMAX-1 | OLS (LinearRegression) | Filtered (16) |
| SARIMAX-2 | OLS | PCA components (9+4) |
| SARIMAX-3 | Ridge (λ=tuned) | All (18) |
| SARIMAX-4 | Elastic Net (α, λ=tuned) | Selected (12) |

**Same SARIMA order for all variants** — order selected by AIC on SARIMAX-1 residuals.

**Note on exogenous values at forecast time:** Evaluation uses *actual* future exogenous values (oracle scenario). This isolates model quality from predictor uncertainty, which is standard in energy forecasting literature.

In [ ]:
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

HORIZONS = [1, 3, 7, 30, 90, 180]
MAX_H    = max(HORIZONS)
SEED     = 42
print('Imports OK')

## Load data and strategy outputs

In [ ]:
# ── Raw splits ──────────────────────────────────────────────────────────────
train = pd.read_csv('data/train.csv', parse_dates=['date'])
val   = pd.read_csv('data/val.csv',   parse_dates=['date'])
test  = pd.read_csv('data/test.csv',  parse_dates=['date'])

TARGET   = 'demand_MW'
y_train  = train[TARGET].values
y_val    = val[TARGET].values
y_test   = test[TARGET].values

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')
print(f'Test  period: {test.date.min().date()} → {test.date.max().date()}')

# ── Strategy feature matrices (scaled continuous + raw calendar) ────────────
def load_strategy(name, split):
    df = pd.read_csv(f'data/strategy_{name}_{split}.csv', parse_dates=['date'])
    return df.drop(columns=['date']).values

X_filtered  = {s: load_strategy('filtered', s)  for s in ['train', 'val', 'test']}
X_pca       = {s: load_strategy('pca', s)       for s in ['train', 'val', 'test']}
X_ridge     = {s: load_strategy('ridge', s)     for s in ['train', 'val', 'test']}
X_elasticnet= {s: load_strategy('elasticnet', s)for s in ['train', 'val', 'test']}

# ── Hyperparameters from notebook 02 ───────────────────────────────────────
with open('data/strategy_ridge_params.json') as f:
    ridge_params = json.load(f)
with open('data/strategy_elasticnet_params.json') as f:
    enet_params = json.load(f)

LAMBDA_RIDGE  = ridge_params['lambda']
ALPHA_ENET    = enet_params['alpha']
L1_RATIO_ENET = enet_params['l1_ratio']

print(f'\nRidge λ = {LAMBDA_RIDGE}')
print(f'ElasticNet α={L1_RATIO_ENET}, λ={ALPHA_ENET}')
print(f'\nStrategy feature dimensions:')
for name, X in [('Filtered', X_filtered), ('PCA', X_pca),
                ('Ridge', X_ridge), ('ElasticNet', X_elasticnet)]:
    print(f'  {name}: {X["train"].shape[1]} features')

---
## SARIMA Order Selection

**Approach:** Fit OLS on filtered features (Strategy 1) to obtain training residuals. Run AIC search over candidate SARIMA orders on those residuals. The selected order is then fixed for all four variants.

**Seasonal period:** s = 7 (weekly). Dutch electricity demand shows a dominant 7-day pattern (weekday/weekend cycle).

**Candidate grid:** 6 non-seasonal × fixed seasonal (0,1,1)_7 = 6 models.

In [ ]:
# ── ADF stationarity test on raw daily demand ──────────────────────────────
adf_result = adfuller(y_train, autolag='AIC')
print(f'ADF test on training demand:')
print(f'  Statistic : {adf_result[0]:.4f}')
print(f'  p-value   : {adf_result[1]:.4f}')
print(f'  → {"Stationary (p<0.05)" if adf_result[1] < 0.05 else "Non-stationary (p≥0.05)"}')

# ── Build strategy 1 residuals (OLS on filtered features, training data) ───
ols_s1 = LinearRegression().fit(X_filtered['train'], y_train)
resid_s1_train = y_train - ols_s1.predict(X_filtered['train'])

adf_resid = adfuller(resid_s1_train, autolag='AIC')
print(f'\nADF test on OLS residuals (Strategy 1):')
print(f'  Statistic : {adf_resid[0]:.4f}')
print(f'  p-value   : {adf_resid[1]:.4f}')
print(f'  Residual mean={resid_s1_train.mean():.2f}, std={resid_s1_train.std():.1f} MW')
print(f'  → {"Stationary" if adf_resid[1] < 0.05 else "Non-stationary"}')

In [ ]:
# ── ACF / PACF on OLS residuals ────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

plot_acf(resid_s1_train, lags=35, ax=axes[0], title='ACF — OLS Residuals (Strategy 1 training)')
plot_pacf(resid_s1_train, lags=35, ax=axes[1], title='PACF — OLS Residuals (Strategy 1 training)')

for ax in axes:
    for lag in [7, 14, 21, 28]:
        ax.axvline(lag, color='orange', alpha=0.5, lw=0.8, ls='--')

plt.tight_layout()
plt.savefig('plots/sarima_acf_pacf.png', bbox_inches='tight')
plt.show()
print('Saved: plots/sarima_acf_pacf.png')
print()
print('Orange dashed lines mark multiples of 7 (weekly seasonality lags).')
print('Significant spikes at lag 7 → seasonal differencing (D=1) or seasonal AR/MA needed.')

In [ ]:
# ── AIC grid search over candidate SARIMA orders ──────────────────────────
# Fixed seasonal component: (P,D,Q,s) = (0,1,1,7)
# Grid over non-seasonal (p,d,q)

SEASONAL_ORDER = (0, 1, 1, 7)

CANDIDATES = [
    (1, 0, 0),
    (0, 0, 1),
    (1, 0, 1),
    (2, 0, 1),
    (1, 1, 1),
    (0, 1, 1),
]

aic_results = []
print(f'Fitting {len(CANDIDATES)} candidate models on Strategy 1 residuals...')
print(f'Seasonal order fixed: {SEASONAL_ORDER}')
print()

for order in CANDIDATES:
    t0 = time.time()
    try:
        m = SARIMAX(resid_s1_train, order=order, seasonal_order=SEASONAL_ORDER,
                    enforce_stationarity=False, enforce_invertibility=False)
        r = m.fit(disp=False)
        aic_results.append({'order': str(order), 'AIC': r.aic, 'BIC': r.bic,
                             'fit_time_s': round(time.time() - t0, 1)})
        print(f'  SARIMA{order}x{SEASONAL_ORDER}  AIC={r.aic:.1f}  BIC={r.bic:.1f}  [{time.time()-t0:.1f}s]')
    except Exception as e:
        print(f'  SARIMA{order}x{SEASONAL_ORDER}  FAILED: {e}')

aic_df = pd.DataFrame(aic_results).sort_values('AIC').reset_index(drop=True)
print()
print('Ranked by AIC:')
display(aic_df)

In [ ]:
# ── Select optimal SARIMA order ────────────────────────────────────────────
import ast
best_order_str = aic_df.iloc[0]['order']
BEST_ORDER = ast.literal_eval(best_order_str)

print(f'Selected SARIMA order : {BEST_ORDER}')
print(f'Seasonal order        : {SEASONAL_ORDER}')
print(f'Best AIC              : {aic_df.iloc[0]["AIC"]:.1f}')
print()

# Check AIC difference to second-best (parsimony check)
delta_aic = aic_df.iloc[1]['AIC'] - aic_df.iloc[0]['AIC']
print(f'ΔAIC to 2nd-best order: {delta_aic:.1f}')
if delta_aic < 2:
    print('  Note: ΔAIC < 2 → models are essentially equivalent; prefer simpler.')
else:
    print('  ΔAIC ≥ 2 → selected order has meaningful AIC advantage.')

---
## Two-Stage Model Fitting

For each strategy:
1. Fit the Stage 1 regression on training data → coefficients β  
2. Compute residuals: `e_train = y_train − X_train · β`  
3. Fit `SARIMA(BEST_ORDER)(SEASONAL_ORDER)` on `e_train`

All four variants share the same SARIMA order, isolating the contribution of the predictor strategy.

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def nrmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return float(rmse / (y_true.max() - y_true.min()) * 100)  # as %


def fit_two_stage(X_tr, y_tr, stage1_model):
    """
    Fit Stage 1 regression and SARIMA on residuals.
    Returns (stage1_model_fitted, sarima_results, y_train_resid).
    """
    stage1 = stage1_model.fit(X_tr, y_tr)
    resid_tr = y_tr - stage1.predict(X_tr)

    sarima_m = SARIMAX(
        resid_tr, order=BEST_ORDER, seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False, enforce_invertibility=False
    )
    sarima_r = sarima_m.fit(disp=False)

    return stage1, sarima_r, resid_tr


def rolling_evaluate(stage1, sarima_r, X_test, y_test, X_test_for_exog,
                     y_test_resid, horizons=HORIZONS, eval_step=30):
    """
    Rolling-origin multi-step evaluation.
    At each origin (every eval_step days), forecast h steps ahead for all h in horizons.
    Exogenous contribution at forecast time uses ACTUAL future X values (oracle scenario).

    Returns dict: {h: {'mape': float, 'nrmse': float, 'n': int}}
    """
    results = {h: {'preds': [], 'actuals': []} for h in horizons}

    # Exogenous demand contribution on test set
    exog_contrib_test = stage1.predict(X_test_for_exog)

    for origin in range(0, len(y_test_resid) - max(horizons), eval_step):
        # Extend SARIMA state to include test residuals up to origin
        if origin == 0:
            res_ext = sarima_r
        else:
            res_ext = sarima_r.append(endog=y_test_resid[:origin], refit=False)

        # Forecast MAX_H residual steps ahead
        fc_resid = res_ext.forecast(steps=max(horizons))

        for h in horizons:
            target_idx = origin + h
            if target_idx >= len(y_test_resid):
                continue
            pred_resid_h = fc_resid[h - 1]
            pred_demand  = pred_resid_h + exog_contrib_test[target_idx]
            actual_demand = y_test[target_idx]

            results[h]['preds'].append(pred_demand)
            results[h]['actuals'].append(actual_demand)

    metrics = {}
    for h in horizons:
        p = np.array(results[h]['preds'])
        a = np.array(results[h]['actuals'])
        metrics[h] = {
            'mape': mape(a, p),
            'nrmse': nrmse(a, p),
            'n_eval': len(p)
        }

    return metrics


print('Helper functions defined.')

In [ ]:
# ── Strategy 1: OLS on Correlation-Filtered Features ──────────────────────
print('Fitting SARIMAX-1 (Correlation Filter + OLS)...')
t0 = time.time()

s1_stage1, s1_sarima, s1_resid_train = fit_two_stage(
    X_filtered['train'], y_train, LinearRegression()
)

print(f'  Stage 1 R²    : {s1_stage1.score(X_filtered["train"], y_train):.4f}')
print(f'  SARIMA AIC    : {s1_sarima.aic:.1f}')
print(f'  Residual std  : {s1_resid_train.std():.1f} MW')
print(f'  Fit time      : {time.time()-t0:.1f}s')

In [ ]:
# ── Strategy 2: OLS on PCA Components ────────────────────────────────────
print('Fitting SARIMAX-2 (PCA + OLS)...')
t0 = time.time()

s2_stage1, s2_sarima, s2_resid_train = fit_two_stage(
    X_pca['train'], y_train, LinearRegression()
)

print(f'  Stage 1 R²    : {s2_stage1.score(X_pca["train"], y_train):.4f}')
print(f'  SARIMA AIC    : {s2_sarima.aic:.1f}')
print(f'  Residual std  : {s2_resid_train.std():.1f} MW')
print(f'  Fit time      : {time.time()-t0:.1f}s')

In [ ]:
# ── Strategy 3: Ridge Regression (L2) + SARIMA ───────────────────────────
print(f'Fitting SARIMAX-3 (Ridge λ={LAMBDA_RIDGE})...')
t0 = time.time()

s3_stage1, s3_sarima, s3_resid_train = fit_two_stage(
    X_ridge['train'], y_train,
    Ridge(alpha=LAMBDA_RIDGE, fit_intercept=True)
)

print(f'  Stage 1 R²    : {s3_stage1.score(X_ridge["train"], y_train):.4f}')
print(f'  SARIMA AIC    : {s3_sarima.aic:.1f}')
print(f'  Residual std  : {s3_resid_train.std():.1f} MW')
print(f'  Fit time      : {time.time()-t0:.1f}s')

In [ ]:
# ── Strategy 4: Elastic Net + SARIMA ─────────────────────────────────────
print(f'Fitting SARIMAX-4 (ElasticNet l1_ratio={L1_RATIO_ENET}, α={ALPHA_ENET})...')
t0 = time.time()

# Note: ElasticNet was fitted on scaled features in notebook 02
# Here we use unscaled features — refit with same hyperparameters
s4_stage1, s4_sarima, s4_resid_train = fit_two_stage(
    X_elasticnet['train'], y_train,
    ElasticNet(alpha=ALPHA_ENET, l1_ratio=L1_RATIO_ENET,
               max_iter=5000, fit_intercept=True)
)

print(f'  Stage 1 R²    : {s4_stage1.score(X_elasticnet["train"], y_train):.4f}')
print(f'  SARIMA AIC    : {s4_sarima.aic:.1f}')
print(f'  Residual std  : {s4_resid_train.std():.1f} MW')
print(f'  Fit time      : {time.time()-t0:.1f}s')

In [ ]:
# ── Stage 1 residual comparison (training) ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
configs = [
    (axes[0,0], s1_resid_train, 'SARIMAX-1 (Corr. Filter)', 'steelblue'),
    (axes[0,1], s2_resid_train, 'SARIMAX-2 (PCA)',          'darkorange'),
    (axes[1,0], s3_resid_train, 'SARIMAX-3 (Ridge)',        'seagreen'),
    (axes[1,1], s4_resid_train, 'SARIMAX-4 (ElasticNet)',   'purple'),
]

dates = train['date'].values
for ax, resid, title, color in configs:
    ax.plot(dates, resid, color=color, alpha=0.6, lw=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{title}\nstd={resid.std():.0f} MW, R²={1-(resid.var()/y_train.var()):.3f}')
    ax.set_ylabel('Residual (MW)')

plt.suptitle('Stage 1 Regression Residuals — Training Set', y=1.01)
plt.tight_layout()
plt.savefig('plots/stage1_residuals_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: plots/stage1_residuals_comparison.png')

---
## Multi-Step Forecast Evaluation

**Protocol:**
- **Rolling-origin evaluation** every 30 days through the test set → ~30 evaluation points per horizon  
- At each origin t, extend the SARIMA state (Kalman filter update, no refitting) and forecast h = 1, 3, 7, 30, 90, 180 steps ahead
- Actual future exogenous values are used at forecast time (oracle exog — isolates temporal forecasting error from predictor uncertainty)
- Metrics: MAPE (%) and nRMSE (%)

In [ ]:
# ── Compute test residuals for each strategy ──────────────────────────────
# These are passed to rolling_evaluate() so the SARIMA state update
# conditions on actual demand residuals (not predicted).

s1_resid_test = y_test - s1_stage1.predict(X_filtered['test'])
s2_resid_test = y_test - s2_stage1.predict(X_pca['test'])
s3_resid_test = y_test - s3_stage1.predict(X_ridge['test'])
s4_resid_test = y_test - s4_stage1.predict(X_elasticnet['test'])

print('Test residual statistics (MW):')
for name, r in [('S1-Filter', s1_resid_test), ('S2-PCA', s2_resid_test),
                ('S3-Ridge', s3_resid_test),   ('S4-ElasticNet', s4_resid_test)]:
    print(f'  {name}: mean={r.mean():.1f}, std={r.std():.1f}')

In [ ]:
# ── Evaluate SARIMAX-1 ─────────────────────────────────────────────────────
print('Evaluating SARIMAX-1 (Correlation Filter)...')
t0 = time.time()
metrics_s1 = rolling_evaluate(
    s1_stage1, s1_sarima,
    X_filtered['test'], y_test,
    X_filtered['test'], s1_resid_test
)
print(f'  Done in {time.time()-t0:.1f}s')
for h, m in metrics_s1.items():
    print(f'  h={h:3d}d: MAPE={m["mape"]:.2f}%  nRMSE={m["nrmse"]:.2f}%  (n={m["n_eval"]})')

In [ ]:
# ── Evaluate SARIMAX-2 ─────────────────────────────────────────────────────
print('Evaluating SARIMAX-2 (PCA)...')
t0 = time.time()
metrics_s2 = rolling_evaluate(
    s2_stage1, s2_sarima,
    X_pca['test'], y_test,
    X_pca['test'], s2_resid_test
)
print(f'  Done in {time.time()-t0:.1f}s')
for h, m in metrics_s2.items():
    print(f'  h={h:3d}d: MAPE={m["mape"]:.2f}%  nRMSE={m["nrmse"]:.2f}%  (n={m["n_eval"]})')

In [ ]:
# ── Evaluate SARIMAX-3 ─────────────────────────────────────────────────────
print('Evaluating SARIMAX-3 (Ridge)...')
t0 = time.time()
metrics_s3 = rolling_evaluate(
    s3_stage1, s3_sarima,
    X_ridge['test'], y_test,
    X_ridge['test'], s3_resid_test
)
print(f'  Done in {time.time()-t0:.1f}s')
for h, m in metrics_s3.items():
    print(f'  h={h:3d}d: MAPE={m["mape"]:.2f}%  nRMSE={m["nrmse"]:.2f}%  (n={m["n_eval"]})')

In [ ]:
# ── Evaluate SARIMAX-4 ─────────────────────────────────────────────────────
print('Evaluating SARIMAX-4 (Elastic Net)...')
t0 = time.time()
metrics_s4 = rolling_evaluate(
    s4_stage1, s4_sarima,
    X_elasticnet['test'], y_test,
    X_elasticnet['test'], s4_resid_test
)
print(f'  Done in {time.time()-t0:.1f}s')
for h, m in metrics_s4.items():
    print(f'  h={h:3d}d: MAPE={m["mape"]:.2f}%  nRMSE={m["nrmse"]:.2f}%  (n={m["n_eval"]})')

---
## Results Tables

In [ ]:
# ── Build consolidated results tables ─────────────────────────────────────
MODELS = {
    'SARIMAX-1 (Filter)': metrics_s1,
    'SARIMAX-2 (PCA)':    metrics_s2,
    'SARIMAX-3 (Ridge)':  metrics_s3,
    'SARIMAX-4 (ElasNet)':metrics_s4,
}

# Load baseline results from notebook 01
baselines = pd.read_csv('data/baseline_results.csv')
baseline_dict = baselines.set_index('Horizon (days)').to_dict()

rows_mape  = []
rows_nrmse = []

# Add baselines
for bl_name, col in [('Seasonal Naive', 'SeasonalNaive MAPE (%)'),
                      ('OLS-Temperature', 'OLS-Temp MAPE (%)')]:
    row = {'Model': bl_name}
    row_n = {'Model': bl_name}
    for h in HORIZONS:
        if h in baseline_dict.get(col, {}):
            row[f'h={h}d'] = round(baseline_dict[col][h], 2)
        else:
            row[f'h={h}d'] = None
        if h in baseline_dict.get('SeasonalNaive nRMSE (%)', {}):
            row_n[f'h={h}d'] = round(baseline_dict['SeasonalNaive nRMSE (%)'][h], 2) if bl_name == 'Seasonal Naive' else None
    rows_mape.append(row)
    rows_nrmse.append(row_n)

# Add SARIMAX models
for model_name, metrics in MODELS.items():
    row_m = {'Model': model_name}
    row_n = {'Model': model_name}
    for h in HORIZONS:
        row_m[f'h={h}d'] = round(metrics[h]['mape'],  2)
        row_n[f'h={h}d'] = round(metrics[h]['nrmse'], 2)
    rows_mape.append(row_m)
    rows_nrmse.append(row_n)

mape_df  = pd.DataFrame(rows_mape)
nrmse_df = pd.DataFrame(rows_nrmse)

print('=== MAPE (%) — lower is better ===')
display(mape_df.set_index('Model'))
print()
print('=== nRMSE (%) — lower is better ===')
display(nrmse_df.set_index('Model'))

In [ ]:
# ── MAPE vs Horizon line plot ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['steelblue', 'darkorange', 'seagreen', 'purple']
styles = ['-o', '-s', '-^', '-D']

for ax, df, ylabel, title in [
    (axes[0], mape_df,  'MAPE (%)',  'MAPE vs Forecast Horizon'),
    (axes[1], nrmse_df, 'nRMSE (%)', 'nRMSE vs Forecast Horizon'),
]:
    h_cols = [f'h={h}d' for h in HORIZONS]

    # Plot baselines with dashed lines
    for i, row in df[df['Model'].str.contains('Naive|OLS')].iterrows():
        vals = [row[c] for c in h_cols]
        if any(v is not None for v in vals):
            ax.plot(HORIZONS, vals, '--', alpha=0.7, label=row['Model'], lw=1.5)

    # Plot SARIMAX models
    sarimax_rows = df[~df['Model'].str.contains('Naive|OLS')]
    for (_, row), color, style in zip(sarimax_rows.iterrows(), colors, styles):
        vals = [row[c] for c in h_cols]
        ax.plot(HORIZONS, vals, style, color=color, label=row['Model'],
                lw=2, ms=7)

    ax.set_xlabel('Forecast Horizon (days)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(HORIZONS)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('SARIMAX Variants vs Baselines — Test Set Performance', y=1.01)
plt.tight_layout()
plt.savefig('plots/sarimax_mape_vs_horizon.png', bbox_inches='tight')
plt.show()
print('Saved: plots/sarimax_mape_vs_horizon.png')

In [ ]:
# ── Forecast vs actual — last 90 days of test set (h=1, each strategy) ────
# Use the rolling eval predictions at h=1 to reconstruct forecast trajectories

def get_h1_predictions(stage1, sarima_r, X_te, y_te, resid_te):
    """Get day-by-day 1-step ahead predictions for the test set."""
    # Extend SARIMA to include all test residuals
    sarima_ext = sarima_r.append(endog=resid_te, refit=False)
    # In-sample predictions for the appended data (true 1-step ahead)
    fitted = sarima_ext.fittedvalues
    resid_pred = fitted[-len(y_te):]  # last N = test period
    exog_pred  = stage1.predict(X_te)
    return resid_pred + exog_pred

PLOT_DAYS = 90
test_dates = test['date'].values[-PLOT_DAYS:]
actual_tail = y_test[-PLOT_DAYS:]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
configs = [
    (axes[0,0], s1_stage1, s1_sarima, X_filtered, s1_resid_test, 'SARIMAX-1 (Filter)',  'steelblue'),
    (axes[0,1], s2_stage1, s2_sarima, X_pca,      s2_resid_test, 'SARIMAX-2 (PCA)',     'darkorange'),
    (axes[1,0], s3_stage1, s3_sarima, X_ridge,    s3_resid_test, 'SARIMAX-3 (Ridge)',   'seagreen'),
    (axes[1,1], s4_stage1, s4_sarima, X_elasticnet,s4_resid_test,'SARIMAX-4 (ElasNet)', 'purple'),
]

for ax, stage1, sarima_r, X_dict, resid_te, name, color in configs:
    preds = get_h1_predictions(stage1, sarima_r, X_dict['test'], y_test, resid_te)
    ax.plot(test_dates, actual_tail, color='gray', lw=1.2, alpha=0.8, label='Actual')
    ax.plot(test_dates, preds[-PLOT_DAYS:], color=color, lw=1.5, alpha=0.9, label='Predicted (h=1)')
    mape_1 = mape(actual_tail, preds[-PLOT_DAYS:])
    ax.set_title(f'{name}  (1-step MAPE: {mape_1:.2f}%)')
    ax.set_ylabel('Demand (MW)')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Forecast vs Actual — Last 90 Test Days (1-step ahead)', y=1.01)
plt.tight_layout()
plt.savefig('plots/sarimax_forecast_vs_actual.png', bbox_inches='tight')
plt.show()
print('Saved: plots/sarimax_forecast_vs_actual.png')

---
## SARIMA Residual Diagnostics

Check whether the SARIMA component has adequately captured temporal autocorrelation. Ljung-Box test on model residuals (null: no autocorrelation up to lag k).

In [ ]:
# ── Ljung-Box test on SARIMA residuals ────────────────────────────────────
print('Ljung-Box test (lag=14) on SARIMA residuals:')
print(f'  Null hypothesis: no autocorrelation in residuals')
print(f'  p > 0.05 → fail to reject null → residuals are white noise (desired)')
print()

for name, sarima_r in [
    ('SARIMAX-1 (Filter)',  s1_sarima),
    ('SARIMAX-2 (PCA)',     s2_sarima),
    ('SARIMAX-3 (Ridge)',   s3_sarima),
    ('SARIMAX-4 (ElasNet)', s4_sarima),
]:
    res_resid = sarima_r.resid
    lb = acorr_ljungbox(res_resid, lags=[7, 14], return_df=True)
    p7  = lb['lb_pvalue'].iloc[0]
    p14 = lb['lb_pvalue'].iloc[1]
    status7  = '✓' if p7  > 0.05 else '✗'
    status14 = '✓' if p14 > 0.05 else '✗'
    print(f'  {name}: lag=7 p={p7:.3f}{status7}  lag=14 p={p14:.3f}{status14}')

# Residual ACF plots for all four models
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (name, sarima_r) in zip(
    axes.flatten(),
    [('SARIMAX-1 (Filter)', s1_sarima), ('SARIMAX-2 (PCA)', s2_sarima),
     ('SARIMAX-3 (Ridge)', s3_sarima),  ('SARIMAX-4 (ElasNet)', s4_sarima)]
):
    plot_acf(sarima_r.resid, lags=28, ax=ax,
             title=f'ACF — {name} residuals')
    for lag in [7, 14, 21, 28]:
        ax.axvline(lag, color='orange', alpha=0.5, lw=0.8, ls='--')

plt.tight_layout()
plt.savefig('plots/sarimax_residual_acf.png', bbox_inches='tight')
plt.show()
print('\nSaved: plots/sarimax_residual_acf.png')

---
## Stage 1 Coefficient Analysis

The key interpretability comparison: how do coefficient signs and magnitudes differ across strategies? This feeds directly into notebook 04's coefficient stability analysis.

In [ ]:
# ── Stage 1 coefficients: Filter vs Ridge vs ElasticNet ───────────────────
# PCA excluded (components not interpretable in original feature space)

# Get feature names for filtered and all-feature strategies
filtered_df = pd.read_csv('data/strategy_filtered_train.csv')
ridge_df    = pd.read_csv('data/strategy_ridge_train.csv')
enet_df     = pd.read_csv('data/strategy_elasticnet_train.csv')

filtered_features = [c for c in filtered_df.columns if c != 'date']
ridge_features    = [c for c in ridge_df.columns if c != 'date']
enet_features     = [c for c in enet_df.columns if c != 'date']

coef_s1 = dict(zip(filtered_features, s1_stage1.coef_))
coef_s3 = dict(zip(ridge_features,    s3_stage1.coef_))
coef_s4 = dict(zip(enet_features,     s4_stage1.coef_))

# Build comparison dataframe for features common across strategies
all_feats = sorted(set(coef_s1) | set(coef_s3) | set(coef_s4))
coef_compare = pd.DataFrame({
    'Feature': all_feats,
    'Filter (OLS)':  [coef_s1.get(f, np.nan) for f in all_feats],
    'Ridge':         [coef_s3.get(f, np.nan) for f in all_feats],
    'ElasticNet':    [coef_s4.get(f, np.nan) for f in all_feats],
}).set_index('Feature').round(2)

# Mark features removed by filtering (NaN) and zeroed by ElasticNet (0.0)
print('Stage 1 Coefficient Comparison (original feature space):')
print('  NaN = feature removed by strategy | 0.00 = zeroed by ElasticNet')
display(coef_compare)

# Sign consistency check against physical priors
print()
print('Physical sign checks (expected: temp_c=−, nao=−, price_eur_kwh=−):')
for feat, expected_neg in [('temp_c', True), ('nao', True), ('price_eur_kwh', True)]:
    for strategy, coef_dict in [('Filter', coef_s1), ('Ridge', coef_s3), ('ElasticNet', coef_s4)]:
        c = coef_dict.get(feat)
        if c is not None and not np.isnan(c):
            sign_ok = (c < 0) == expected_neg
            print(f'  {strategy} {feat}: {c:.2f} {"✓" if sign_ok else "✗ (unexpected sign)"}')

---
## Save Results

In [ ]:
# ── Save all evaluation results ────────────────────────────────────────────
import os

# Full results table (MAPE and nRMSE)
all_rows = []
for model_name, metrics in MODELS.items():
    for h in HORIZONS:
        all_rows.append({
            'model': model_name,
            'horizon_days': h,
            'mape_pct': round(metrics[h]['mape'], 4),
            'nrmse_pct': round(metrics[h]['nrmse'], 4),
            'n_eval_points': metrics[h]['n_eval']
        })

results_df = pd.DataFrame(all_rows)
results_df.to_csv('data/sarimax_results.csv', index=False)

# MAPE and nRMSE tables
mape_df.to_csv('data/sarimax_mape_table.csv', index=False)
nrmse_df.to_csv('data/sarimax_nrmse_table.csv', index=False)

# SARIMA order
order_info = {
    'order': list(BEST_ORDER),
    'seasonal_order': list(SEASONAL_ORDER),
    'selection_criterion': 'AIC',
    'aic': float(aic_df.iloc[0]['AIC'])
}
with open('data/sarima_order.json', 'w') as f:
    json.dump(order_info, f, indent=2)

# Stage 1 coefficients
coef_compare.to_csv('data/sarimax_stage1_coefs.csv')

# Summary: best model per horizon
best_per_h = results_df.loc[results_df.groupby('horizon_days')['mape_pct'].idxmin(),
                             ['horizon_days','model','mape_pct','nrmse_pct']]
best_per_h.to_csv('data/sarimax_best_per_horizon.csv', index=False)

outputs = ['data/sarimax_results.csv', 'data/sarimax_mape_table.csv',
           'data/sarimax_nrmse_table.csv', 'data/sarima_order.json',
           'data/sarimax_stage1_coefs.csv', 'data/sarimax_best_per_horizon.csv']

print('Saved outputs:')
for f in outputs:
    size = os.path.getsize(f) // 1024
    print(f'  ✓  {size:>3} KB  {f}')

print()
print('Best SARIMAX model per horizon (by MAPE):')
display(best_per_h)

---
## Next step → `04_interpretability.ipynb`

Notebook 04 uses the SARIMAX coefficient estimates from this notebook to:
1. **Coefficient stability analysis** — 5-fold TimeSeriesSplit CV; Coefficient of Variation (CV = std/|mean|) and sign consistency per predictor per strategy
2. **SHAP analysis on residuals** — XGBoost fitted on SARIMAX residuals; quantifies information discarded by filtering/zeroing
3. **H1, H2, H3 hypothesis testing** — quantitative comparison of accuracy and stability across strategies

Key inputs from this notebook: `data/sarimax_stage1_coefs.csv`, `data/sarimax_results.csv`, and the four `(stage1, sarima)` model objects (to be re-fitted in notebook 04).